# Business Performance Analytics

This project compares six technology companies on operating performance. The SQL calculates the metrics. The written analysis comes after the results, once for each company, and the management suggestions come after those company reads.

**Business question**

How is each company performing across growth, operations, profitability, cash generation, and financial health, and where should management attention be focused?

**Companies**

| Company | Sector |
|---|---|
| Microsoft | Software & Cloud |
| Apple | Consumer Technology |
| NVIDIA | Semiconductors |
| Adobe | Software |
| AMD | Semiconductors |
| CrowdStrike | Cybersecurity |

**Fiscal-year coverage**

Latest year is not the same calendar year for every company. Microsoft, NVIDIA, and CrowdStrike have a FY2026 10-K. AMD, Adobe, and Apple stop at FY2025. FY2025 is the last year available for all six companies. A FY2026 comparison drops AMD, Adobe, and Apple. Company analyses use each company's own latest year. Peer ranks stay on FY2025.

**How to read this notebook**

1. Growth and momentum
2. Profitability
3. Operating efficiency
4. Cash flow and cash conversion
5. Financial health
6. Peer benchmarking for FY2025
7. Company analysis
8. Management suggestions

**Pipeline**

SEC 10-K filings, Python preparation, `financial_analytics.clean.company_financials`, SQL views, then the company analysis.

Views used: `growth_trend`, `performance`, `financial_health`, `peer_benchmarking`, and `executive_summary`.


## Data Preparation & Quality

### Business Question

Is the data complete enough to support the operating metrics?

Source table: `financial_analytics.clean.company_financials`

Missing values are left null. They are not filled with zero.

AMD Total Liabilities was missing in the source file. Where needed it is calculated as:

Total Liabilities = Total Assets − Stockholders’ Equity

`extract_sec_company_facts.py` writes `data/raw/sec_company_financials_raw.csv`, `data/processed/sec_company_financials_clean.csv`, and `data/processed/company_financials_wide.csv`. This notebook starts from `financial_analytics.clean.company_financials` and `financial_analytics.clean.sec_company_financials_clean`, which are the warehouse copies of those prepared files. The upload step is not in this notebook.


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS financial_analytics.data_quality;

CREATE OR REPLACE VIEW financial_analytics.data_quality.sec_company_financials_null_summary AS

SELECT
    COUNT(*) AS total_records,
    SUM(CASE WHEN company IS NULL THEN 1 ELSE 0 END) AS company_nulls,
    SUM(CASE WHEN ticker IS NULL THEN 1 ELSE 0 END) AS ticker_nulls,
    SUM(CASE WHEN statement IS NULL THEN 1 ELSE 0 END) AS statement_nulls,
    SUM(CASE WHEN fiscal_year IS NULL THEN 1 ELSE 0 END) AS fiscal_year_nulls,
    SUM(CASE WHEN Metric IS NULL THEN 1 ELSE 0 END) AS Metric_nulls,
    SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) AS value_nulls,
    ROUND(SUM(CASE WHEN company IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS company_null_pct,
    ROUND(SUM(CASE WHEN ticker IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS ticker_null_pct,
    ROUND(SUM(CASE WHEN statement IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS statement_null_pct,
    ROUND(SUM(CASE WHEN fiscal_year IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS fiscal_year_null_pct,
    ROUND(SUM(CASE WHEN Metric IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Metric_null_pct,
    ROUND(SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS value_null_pct
FROM financial_analytics.clean.sec_company_financials_clean;


In [0]:
%sql
SELECT * FROM financial_analytics.data_quality.sec_company_financials_null_summary;


total_records,company_nulls,ticker_nulls,statement_nulls,fiscal_year_nulls,Metric_nulls,value_nulls,company_null_pct,ticker_null_pct,statement_null_pct,fiscal_year_null_pct,Metric_null_pct,value_null_pct
1657,0,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.00


In [0]:
%sql
-- This UPDATE fills AMD nulls on the wide table for the dollar check.
-- Total_Liabilities_Imputed does not use that null. The financial_health view
-- reads sec_company_financials_clean and flags AMD years with no
-- Total_Liabilities fact, including after this fill.
Update financial_analytics.clean.company_financials

set Total_Liabilities = Total_Assets - Total_Stockholders_Equity
where Company = 'AMD'
and Total_Liabilities is null;


num_affected_rows
5


### Company-year quality checks

Each row is one control. Counts come from the clean table.


In [0]:
%sql
CREATE OR REPLACE VIEW financial_analytics.data_quality.company_year_quality AS
SELECT
    COUNT(*) AS total_company_years,
    SUM(CASE WHEN Revenue IS NULL THEN 1 ELSE 0 END) AS missing_revenue,
    SUM(CASE WHEN Operating_Income IS NULL THEN 1 ELSE 0 END) AS missing_operating_income,
    SUM(CASE WHEN Net_Income IS NULL THEN 1 ELSE 0 END) AS missing_net_income,
    SUM(CASE WHEN Operating_Expenses IS NULL THEN 1 ELSE 0 END) AS missing_operating_expenses,
    SUM(CASE WHEN Cash_Flow_from_Operating_Activities IS NULL THEN 1 ELSE 0 END) AS missing_operating_cash_flow,
    SUM(CASE WHEN Total_Current_Assets IS NULL THEN 1 ELSE 0 END) AS missing_current_assets,
    SUM(CASE WHEN Total_Current_Liabilities IS NULL THEN 1 ELSE 0 END) AS missing_current_liabilities,
    SUM(CASE WHEN Total_Assets IS NULL THEN 1 ELSE 0 END) AS missing_total_assets,
    SUM(CASE WHEN Total_Liabilities IS NULL THEN 1 ELSE 0 END) AS missing_total_liabilities,
    SUM(CASE WHEN Total_Stockholders_Equity IS NULL THEN 1 ELSE 0 END) AS missing_equity,
    SUM(CASE WHEN Total_Stockholders_Equity <= 0 THEN 1 ELSE 0 END) AS zero_or_negative_equity,
    SUM(CASE WHEN Short_Term_Debt IS NULL AND Long_Term_Debt IS NULL THEN 1 ELSE 0 END) AS missing_debt,
    SUM(CASE WHEN Total_Current_Liabilities = 0 THEN 1 ELSE 0 END) AS zero_current_liabilities
FROM financial_analytics.clean.company_financials;


In [0]:
%sql
SELECT *
FROM financial_analytics.data_quality.company_year_quality;


total_company_years,missing_revenue,missing_operating_income,missing_net_income,missing_operating_expenses,missing_operating_cash_flow,missing_current_assets,missing_current_liabilities,missing_total_assets,missing_total_liabilities,missing_equity,zero_or_negative_equity,missing_debt,zero_current_liabilities
33,0,0,0,4,0,0,0,0,0,0,0,0,0


## SQL views




In [0]:
%sql
-- Revenue growth and net income margin.
-- Prior year is the previous fiscal year for the same company.
-- Growth_Differential uses the unrounded growth rates, then each result is rounded to 3 decimals.
CREATE OR REPLACE VIEW financial_analytics.analytics.growth_trend AS
WITH growth AS (
    SELECT
        Company,
        Fiscal_Year,
        Revenue,
        Net_Income,
        (Revenue - LAG(Revenue) OVER (PARTITION BY Company ORDER BY Fiscal_Year))
            / NULLIF(LAG(Revenue) OVER (PARTITION BY Company ORDER BY Fiscal_Year), 0) AS Growth_Rate
    FROM financial_analytics.clean.company_financials
),
growth_metrics AS (
    SELECT
        Company,
        Fiscal_Year,
        Revenue,
        Net_Income,
        Growth_Rate,
        LAG(Growth_Rate) OVER (PARTITION BY Company ORDER BY Fiscal_Year) AS Prior_Growth_Rate,
        Growth_Rate - LAG(Growth_Rate) OVER (PARTITION BY Company ORDER BY Fiscal_Year) AS Growth_Differential
    FROM growth
)
SELECT
    Company,
    Fiscal_Year,
    Revenue,
    Net_Income,
    round(Growth_Rate, 3) AS Growth_Rate,
    round(Prior_Growth_Rate, 3) AS Prior_Growth_Rate,
    round(Growth_Differential, 3) AS Growth_Differential,
    round(Net_Income / NULLIF(Revenue, 0), 3) AS Net_Income_Margin_pct
FROM growth_metrics


In [0]:
%sql
-- Profitability, operating efficiency, and cash conversion.
-- Cash_Conversion_Ratio keeps the raw OCF / Net Income value, rounded to 3 decimals, and is null when net income is 0.
-- Cash_Conversion_Meaningful is false when net income is zero or negative, or when
-- abs(OCF / Net Income) > 5. A ratio of exactly 5 stays meaningful.
-- The threshold comparison uses the unrounded ratio.
-- Revenue_Growth_Minus_Opex_Growth uses both growth rates at full precision and rounds only the difference.
-- Free_Cash_Flow = OCF - Capital Expenditures. Capex is a positive spend.
CREATE OR REPLACE VIEW financial_analytics.analytics.performance AS
WITH base AS (
    SELECT
        c.Company,
        c.Fiscal_Year,
        c.Revenue,
        c.Gross_Profit,
        c.Net_Income,
        c.Operating_Income,
        c.Operating_Expenses,
        c.Cash_Flow_from_Operating_Activities,
        c.Capital_Expenditures,
        LAG(c.Revenue) OVER (PARTITION BY c.Company ORDER BY c.Fiscal_Year) AS Prior_Revenue,
        LAG(c.Operating_Income) OVER (PARTITION BY c.Company ORDER BY c.Fiscal_Year) AS Prior_Operating_Income,
        LAG(c.Net_Income) OVER (PARTITION BY c.Company ORDER BY c.Fiscal_Year) AS Prior_Net_Income,
        LAG(c.Operating_Expenses) OVER (PARTITION BY c.Company ORDER BY c.Fiscal_Year) AS Prior_Opex,
        LAG(c.Cash_Flow_from_Operating_Activities) OVER (PARTITION BY c.Company ORDER BY c.Fiscal_Year) AS Prior_OCF
    FROM financial_analytics.clean.company_financials c
)
SELECT
    b.Company,
    b.Fiscal_Year,
    b.Revenue,
    b.Gross_Profit,
    b.Net_Income,
    b.Operating_Income,
    b.Operating_Expenses,
    b.Cash_Flow_from_Operating_Activities,
    b.Capital_Expenditures,
    gt.Growth_Rate,
    gt.Net_Income_Margin_pct,
    round(b.Gross_Profit / NULLIF(b.Revenue, 0), 3) AS Gross_Margin_pct,
    round(b.Operating_Income / NULLIF(b.Revenue, 0), 3) AS Operating_Margin_pct,
    round((b.Operating_Income - b.Prior_Operating_Income) / NULLIF(b.Prior_Operating_Income, 0), 3) AS Operating_Income_Growth,
    round((b.Net_Income - b.Prior_Net_Income) / NULLIF(b.Prior_Net_Income, 0), 3) AS Net_Income_Growth,
    round((b.Operating_Expenses - b.Prior_Opex) / NULLIF(b.Prior_Opex, 0), 3) AS Operating_Expense_Growth,
    round(b.Operating_Expenses / NULLIF(b.Revenue, 0), 3) AS Operating_Expense_to_Revenue,
    round(
        ((b.Revenue - b.Prior_Revenue) / NULLIF(b.Prior_Revenue, 0))
        - ((b.Operating_Expenses - b.Prior_Opex) / NULLIF(b.Prior_Opex, 0)),
        3
    ) AS Revenue_Growth_Minus_Opex_Growth,
    round(b.Cash_Flow_from_Operating_Activities / NULLIF(b.Revenue, 0), 3) AS Operating_Cash_Flow_Margin_pct,
    round(
        (b.Cash_Flow_from_Operating_Activities - b.Prior_OCF) / NULLIF(b.Prior_OCF, 0),
        3
    ) AS Operating_Cash_Flow_Growth,
    round(b.Cash_Flow_from_Operating_Activities / NULLIF(b.Net_Income, 0), 3) AS Cash_Conversion_Ratio,
    CASE
        WHEN b.Net_Income IS NULL OR b.Cash_Flow_from_Operating_Activities IS NULL THEN NULL
        WHEN b.Net_Income <= 0 THEN FALSE
        WHEN abs(b.Cash_Flow_from_Operating_Activities / b.Net_Income) > 5 THEN FALSE
        ELSE TRUE
    END AS Cash_Conversion_Meaningful,
    round(b.Cash_Flow_from_Operating_Activities - b.Capital_Expenditures, 3) AS Free_Cash_Flow,
    round(
        round(b.Cash_Flow_from_Operating_Activities - b.Capital_Expenditures, 3) / NULLIF(b.Revenue, 0),
        3
    ) AS Free_Cash_Flow_Margin_pct
FROM base b
LEFT JOIN financial_analytics.analytics.growth_trend gt
    ON b.Company = gt.Company
    AND b.Fiscal_Year = gt.Fiscal_Year


In [0]:
%sql
-- Liquidity, leverage, and returns.
-- Total debt coalesces a missing short-term or long-term leg to 0. The reported flags stay false when the source was null.
-- CrowdStrike short-term debt is unreported for FY2021-FY2026 and is treated as 0.
-- AMD total liabilities are imputed as Total Assets - Total Stockholders' Equity when the long facts
-- table has no Total_Liabilities row for that company-year. That is the source provenance.
-- The wide-table UPDATE may already have filled those nulls. The flag still follows the missing fact,
-- not whether company_financials.Total_Liabilities is null now.
-- A reported Total_Liabilities fact is never imputed, even when the amount equals assets minus equity.
-- Other companies are not imputed. If the long facts table cannot be read, do not infer the flag.
-- Primary ROA and ROE use average balances, (current + prior) / 2, partitioned by ticker.
-- Ending balances are used only when there is no prior year. ROA_Ending and ROE_Ending always use the ending balance.
-- ROA_Avg and ROE_Avg are null when there is no prior year.
CREATE OR REPLACE VIEW financial_analytics.analytics.financial_health AS
WITH reported_liabilities AS (
    SELECT DISTINCT
        Company,
        Fiscal_Year
    FROM financial_analytics.clean.sec_company_financials_clean
    WHERE Metric = 'Total_Liabilities'
),
source AS (
    SELECT
        c.Company,
        c.Ticker,
        c.Fiscal_Year,
        c.Total_Assets,
        c.Total_Stockholders_Equity,
        c.Total_Current_Assets,
        c.Total_Current_Liabilities,
        c.Short_Term_Debt,
        c.Long_Term_Debt,
        c.Net_Income,
        c.Cash_Flow_from_Operating_Activities,
        CASE
            WHEN c.Company = 'AMD' AND r.Company IS NULL
                THEN COALESCE(c.Total_Liabilities, c.Total_Assets - c.Total_Stockholders_Equity)
            ELSE c.Total_Liabilities
        END AS Total_Liabilities,
        CASE
            WHEN c.Company = 'AMD' AND r.Company IS NULL THEN TRUE
            ELSE FALSE
        END AS Total_Liabilities_Imputed
    FROM financial_analytics.clean.company_financials c
    LEFT JOIN reported_liabilities r
        ON c.Company = r.Company
        AND c.Fiscal_Year = r.Fiscal_Year
),
with_prior AS (
    SELECT
        source.*,
        LAG(Total_Assets) OVER (PARTITION BY Ticker ORDER BY Fiscal_Year) AS Prior_Total_Assets,
        LAG(Total_Stockholders_Equity) OVER (PARTITION BY Ticker ORDER BY Fiscal_Year) AS Prior_Total_Equity
    FROM source
)
SELECT
    Company,
    Fiscal_Year,
    Total_Assets,
    Total_Liabilities,
    Total_Liabilities_Imputed,
    Total_Stockholders_Equity,
    Total_Current_Assets,
    Total_Current_Liabilities,
    round(Total_Current_Assets / NULLIF(Total_Current_Liabilities, 0), 3) AS Current_ratio,
    COALESCE(Short_Term_Debt, 0) + COALESCE(Long_Term_Debt, 0) AS Total_Debt,
    (Short_Term_Debt IS NOT NULL) AS Short_Term_Debt_Reported,
    (Long_Term_Debt IS NOT NULL) AS Long_Term_Debt_Reported,
    round(
        (COALESCE(Short_Term_Debt, 0) + COALESCE(Long_Term_Debt, 0))
            / NULLIF(Total_Stockholders_Equity, 0),
        3
    ) AS debt_to_equity_ratio,
    round(Total_Liabilities / NULLIF(Total_Assets, 0), 3) AS Liabilities_to_Assets,
    round(Net_Income / NULLIF(Total_Assets, 0), 3) AS ROA_Ending,
    round(Net_Income / NULLIF(Total_Stockholders_Equity, 0), 3) AS ROE_Ending,
    round(
        Net_Income / NULLIF((Total_Assets + Prior_Total_Assets) / 2, 0),
        3
    ) AS ROA_Avg,
    round(
        Net_Income / NULLIF((Total_Stockholders_Equity + Prior_Total_Equity) / 2, 0),
        3
    ) AS ROE_Avg,
    CASE WHEN Prior_Total_Assets IS NULL THEN TRUE ELSE FALSE END AS ROA_Uses_Ending_Balance,
    CASE WHEN Prior_Total_Equity IS NULL THEN TRUE ELSE FALSE END AS ROE_Uses_Ending_Balance,
    round(
        CASE
            WHEN Prior_Total_Assets IS NULL THEN Net_Income / NULLIF(Total_Assets, 0)
            ELSE Net_Income / NULLIF((Total_Assets + Prior_Total_Assets) / 2, 0)
        END,
        3
    ) AS ROA,
    round(
        CASE
            WHEN Prior_Total_Equity IS NULL THEN Net_Income / NULLIF(Total_Stockholders_Equity, 0)
            ELSE Net_Income / NULLIF((Total_Stockholders_Equity + Prior_Total_Equity) / 2, 0)
        END,
        3
    ) AS ROE,
    Cash_Flow_from_Operating_Activities
FROM with_prior


In [0]:
%sql
-- FY2025 peer ranks. FY2025 is the last fiscal year present for every company.
-- The ranked set is exactly these nine metrics. Average_Rank is their equal-weight mean, rounded to 3 decimals.
-- Higher is better: revenue growth, operating margin, net income margin, OCF margin, current ratio, ROA, ROE.
-- Lower is better: operating expense / revenue, debt-to-equity.
-- Ties: RANK() shares the best rank and skips the next rank (pandas rank method='min').
-- Missing values: NULLS LAST, so a missing metric ranks after every reported value. Missing values tie with each other.
-- ROA and ROE are the primary returns (average balances, ending balance only when there is no prior year).
-- The value column sits next to its rank.
CREATE OR REPLACE VIEW financial_analytics.analytics.peer_benchmarking AS
WITH ranked_metrics AS (
    SELECT
        p.Company,
        p.Fiscal_Year,
        gt.Growth_Rate,
        p.Operating_Margin_pct,
        p.Net_Income_Margin_pct,
        p.Operating_Expense_to_Revenue,
        p.Operating_Cash_Flow_Margin_pct,
        fh.Current_ratio,
        fh.debt_to_equity_ratio,
        fh.ROA,
        fh.ROE,
        RANK() OVER (PARTITION BY p.Fiscal_Year ORDER BY gt.Growth_Rate DESC NULLS LAST) AS Growth_Rate_Rank,
        RANK() OVER (PARTITION BY p.Fiscal_Year ORDER BY p.Operating_Margin_pct DESC NULLS LAST) AS Operating_Margin_Rank,
        RANK() OVER (PARTITION BY p.Fiscal_Year ORDER BY p.Net_Income_Margin_pct DESC NULLS LAST) AS Net_Income_Margin_Rank,
        RANK() OVER (PARTITION BY p.Fiscal_Year ORDER BY p.Operating_Expense_to_Revenue ASC NULLS LAST) AS Operating_Expense_to_Revenue_Rank,
        RANK() OVER (PARTITION BY p.Fiscal_Year ORDER BY p.Operating_Cash_Flow_Margin_pct DESC NULLS LAST) AS Operating_Cash_Flow_Margin_Rank,
        RANK() OVER (PARTITION BY p.Fiscal_Year ORDER BY fh.Current_ratio DESC NULLS LAST) AS Current_Ratio_Rank,
        RANK() OVER (PARTITION BY p.Fiscal_Year ORDER BY fh.debt_to_equity_ratio ASC NULLS LAST) AS Debt_to_Equity_Rank,
        RANK() OVER (PARTITION BY p.Fiscal_Year ORDER BY fh.ROA DESC NULLS LAST) AS ROA_Rank,
        RANK() OVER (PARTITION BY p.Fiscal_Year ORDER BY fh.ROE DESC NULLS LAST) AS ROE_Rank
    FROM financial_analytics.analytics.performance p
    LEFT JOIN financial_analytics.analytics.financial_health fh
        ON p.Company = fh.Company
        AND p.Fiscal_Year = fh.Fiscal_Year
    LEFT JOIN financial_analytics.analytics.growth_trend gt
        ON p.Company = gt.Company
        AND p.Fiscal_Year = gt.Fiscal_Year
    WHERE p.Fiscal_Year = 2025
)
SELECT
    Company,
    Fiscal_Year,
    Growth_Rate,
    Growth_Rate_Rank,
    Operating_Margin_pct,
    Operating_Margin_Rank,
    Net_Income_Margin_pct,
    Net_Income_Margin_Rank,
    Operating_Expense_to_Revenue,
    Operating_Expense_to_Revenue_Rank,
    Operating_Cash_Flow_Margin_pct,
    Operating_Cash_Flow_Margin_Rank,
    Current_ratio,
    Current_Ratio_Rank,
    debt_to_equity_ratio,
    Debt_to_Equity_Rank,
    ROA,
    ROA_Rank,
    ROE,
    ROE_Rank,
    round((
        Growth_Rate_Rank
        + Operating_Margin_Rank
        + Net_Income_Margin_Rank
        + Operating_Expense_to_Revenue_Rank
        + Operating_Cash_Flow_Margin_Rank
        + Current_Ratio_Rank
        + Debt_to_Equity_Rank
        + ROA_Rank
        + ROE_Rank
    ) / 9.0, 3) AS Average_Rank
FROM ranked_metrics


In [0]:
%sql
-- Latest fiscal year for each company.
-- FY2025 benchmark ranks are attached on Company alone, including when the latest year is FY2026.
-- Rank columns are labeled FY2025_. Benchmark_Fiscal_Year is 2025 on every row.
-- ROA and ROE are the primary average-balance returns.
CREATE OR REPLACE VIEW financial_analytics.analytics.executive_summary AS
WITH latest AS (
    SELECT Company, MAX(Fiscal_Year) AS Latest_Fiscal_Year
    FROM financial_analytics.analytics.performance
    GROUP BY Company
)
SELECT
    p.Company,
    p.Fiscal_Year,
    2025 AS Benchmark_Fiscal_Year,
    gt.Revenue,
    gt.Growth_Rate,
    gt.Prior_Growth_Rate,
    gt.Growth_Differential,
    p.Gross_Margin_pct,
    p.Operating_Margin_pct,
    p.Net_Income_Margin_pct,
    p.Operating_Income_Growth,
    p.Net_Income_Growth,
    p.Operating_Expense_Growth,
    p.Operating_Expense_to_Revenue,
    p.Revenue_Growth_Minus_Opex_Growth,
    p.Operating_Cash_Flow_Margin_pct,
    p.Operating_Cash_Flow_Growth,
    p.Cash_Conversion_Ratio,
    p.Cash_Conversion_Meaningful,
    p.Free_Cash_Flow,
    p.Free_Cash_Flow_Margin_pct,
    fh.Current_ratio,
    fh.debt_to_equity_ratio,
    fh.Liabilities_to_Assets,
    fh.ROA,
    fh.ROE,
    fh.ROA_Ending,
    fh.ROE_Ending,
    fh.ROA_Avg,
    fh.ROE_Avg,
    fh.ROA_Uses_Ending_Balance,
    fh.ROE_Uses_Ending_Balance,
    fh.Total_Liabilities_Imputed,
    fh.Short_Term_Debt_Reported,
    fh.Long_Term_Debt_Reported,
    pb.Growth_Rate_Rank AS FY2025_Growth_Rate_Rank,
    pb.Operating_Margin_Rank AS FY2025_Operating_Margin_Rank,
    pb.Net_Income_Margin_Rank AS FY2025_Net_Income_Margin_Rank,
    pb.Operating_Expense_to_Revenue_Rank AS FY2025_Operating_Expense_to_Revenue_Rank,
    pb.Operating_Cash_Flow_Margin_Rank AS FY2025_Operating_Cash_Flow_Margin_Rank,
    pb.Current_Ratio_Rank AS FY2025_Current_Ratio_Rank,
    pb.Debt_to_Equity_Rank AS FY2025_Debt_to_Equity_Rank,
    pb.ROA_Rank AS FY2025_ROA_Rank,
    pb.ROE_Rank AS FY2025_ROE_Rank,
    pb.Average_Rank AS FY2025_Average_Rank
FROM financial_analytics.analytics.performance p
INNER JOIN latest
    ON p.Company = latest.Company
    AND p.Fiscal_Year = latest.Latest_Fiscal_Year
LEFT JOIN financial_analytics.analytics.growth_trend gt
    ON p.Company = gt.Company
    AND p.Fiscal_Year = gt.Fiscal_Year
LEFT JOIN financial_analytics.analytics.financial_health fh
    ON p.Company = fh.Company
    AND p.Fiscal_Year = fh.Fiscal_Year
LEFT JOIN financial_analytics.analytics.peer_benchmarking pb
    ON p.Company = pb.Company


## Growth and momentum

### Business question

Is the company growing, and is that growth accelerating or slowing?

### Approach

View: growth_trend

- Growth_Rate = (current revenue − prior revenue) / prior revenue
- Prior_Growth_Rate = previous year's growth rate
- Growth_Differential = current growth rate − prior growth rate


In [0]:
%sql
SELECT Company, fiscal_year, Revenue, Growth_Rate, Prior_Growth_Rate, Growth_Differential
FROM financial_analytics.analytics.growth_trend 
ORDER BY Company, Fiscal_Year;


Company,fiscal_year,Revenue,Growth_Rate,Prior_Growth_Rate,Growth_Differential
AMD,2021,1.6434E10,null,null,null
AMD,2022,2.3601E10,0.436,null,null
AMD,2023,2.268E10,-0.039,0.436,-0.475
AMD,2024,2.5785E10,0.137,-0.039,0.176
AMD,2025,3.4639E10,0.343,0.137,0.206
Adobe,2021,1.5785E10,null,null,null
Adobe,2022,1.7606E10,0.115,null,null
Adobe,2023,1.9409E10,0.102,0.115,-0.013
Adobe,2024,2.1505E10,0.108,0.102,0.006
Adobe,2025,2.3769E10,0.105,0.108,-0.003


In [0]:
%sql
SELECT 
Company,
fiscal_year,
Growth_Differential,
Growth_Rate
FROM financial_analytics.analytics.growth_trend
WHERE fiscal_year >=2024
ORDER BY company


Company,fiscal_year,Growth_Differential,Growth_Rate
AMD,2024,0.176,0.137
AMD,2025,0.206,0.343
Adobe,2024,0.006,0.108
Adobe,2025,-0.003,0.105
Apple,2024,0.048,0.02
Apple,2025,0.044,0.064
CrowdStrike,2024,-0.181,0.363
CrowdStrike,2025,-0.069,0.294
CrowdStrike,2026,-0.077,0.217
Microsoft,2024,0.088,0.157


## Profitability

### Business Question

Is growth turning into profitable operations?

### Approach

View: `performance`

- Gross_Margin_pct = Gross Profit / Revenue
- Operating_Margin_pct = Operating Income / Revenue
- Net_Income_Margin_pct = Net Income / Revenue


In [0]:
%sql
SELECT Company, Fiscal_Year,Gross_Margin_pct ,Net_Income_Margin_pct ,Operating_Margin_pct,Operating_Income_Growth
FROM financial_analytics.analytics.performance
Where Fiscal_Year >= 2024
ORDER BY company, Fiscal_Year;


Company,Fiscal_Year,Gross_Margin_pct,Net_Income_Margin_pct,Operating_Margin_pct,Operating_Income_Growth
AMD,2024,0.494,0.064,0.074,3.738
AMD,2025,0.495,0.125,0.107,0.944
Adobe,2024,0.89,0.259,0.313,0.014
Adobe,2025,0.893,0.3,0.366,0.291
Apple,2024,0.462,0.24,0.315,0.078
Apple,2025,0.469,0.269,0.32,0.08
CrowdStrike,2024,0.752,0.024,-0.006,-0.899
CrowdStrike,2025,0.75,-0.004,-0.029,5.081
CrowdStrike,2026,0.747,-0.034,-0.061,1.52
Microsoft,2024,0.698,0.36,0.446,0.236


## Operating Efficiency

### Business Question

Is the business scaling efficiently, or are operating costs growing faster than revenue?

### Approach

View: `performance`

- Operating_Expense_Growth vs prior year
- Operating_Expense_to_Revenue
- Revenue_Growth_Minus_Opex_Growth = unrounded revenue growth − unrounded operating-expense growth, then rounded to 3 decimals

If revenue growth is higher than opex growth, efficiency is improving. If opex growth is higher, that is cost pressure to investigate.


In [0]:
%sql
SELECT p.Company, p.Fiscal_Year, p.Revenue,gt.Growth_Rate,p.Operating_Expense_Growth, p.Operating_Expense_to_Revenue, p.Revenue_Growth_Minus_Opex_Growth
FROM financial_analytics.analytics.performance p
join financial_analytics.analytics.growth_trend gt
on p.company=gt.company and p.fiscal_year=gt.fiscal_year
where p.Fiscal_Year >= 2024
ORDER BY Company, Fiscal_Year;


Company,Fiscal_Year,Revenue,Growth_Rate,Operating_Expense_Growth,Operating_Expense_to_Revenue,Revenue_Growth_Minus_Opex_Growth
AMD,2024,2.5785E10,0.137,0.076,0.42,0.061
AMD,2025,3.4639E10,0.343,0.243,0.389,0.1
Adobe,2024,2.1505E10,0.108,0.192,0.577,-0.084
Adobe,2025,2.3769E10,0.105,0.009,0.526,0.096
Apple,2024,3.91035E11,0.02,0.048,0.147,-0.028
Apple,2025,4.16161E11,0.064,0.082,0.149,-0.018
CrowdStrike,2024,3.055555E9,0.363,0.265,0.758,0.098
CrowdStrike,2025,3.953624E9,0.294,0.33,0.779,-0.036
CrowdStrike,2026,4.812005E9,0.217,0.262,0.808,-0.045
Microsoft,2024,2.45122E11,0.157,0.07,0.251,0.087


## Cash Flow & Cash Conversion

### Business Question

Is reported performance supported by cash from operations?

### Approach

View: `performance`

- Operating_Cash_Flow_Margin_pct = OCF / Revenue
- Operating_Cash_Flow_Growth vs prior year
- Cash_Conversion_Ratio = OCF / Net Income (null if net income is 0). The raw ratio is kept.
- Cash_Conversion_Meaningful is false when net income is zero or negative, or when abs(OCF / Net Income) > 5. A ratio of exactly 5 stays meaningful.
- Where the flag is false, read the ratio as n/m. CrowdStrike FY2025 is the example: a small net loss against large operating cash flow.
- Free cash flow, operating cash flow minus capital expenditures, is calculated in the view. It is null when capital expenditures are missing, including NVIDIA FY2021. The company reads below use operating cash flow.


In [0]:
%sql
SELECT Company, Fiscal_Year, Net_Income, Cash_Flow_from_Operating_Activities, Operating_Cash_Flow_Margin_pct, Operating_Cash_Flow_Growth, Cash_Conversion_Ratio, Capital_Expenditures, Free_Cash_Flow, Free_Cash_Flow_Margin_pct
FROM financial_analytics.analytics.performance
where Fiscal_Year >= 2024
ORDER BY Company, Fiscal_Year;


Company,Fiscal_Year,Net_Income,Cash_Flow_from_Operating_Activities,Operating_Cash_Flow_Margin_pct,Operating_Cash_Flow_Growth,Cash_Conversion_Ratio,Capital_Expenditures,Free_Cash_Flow,Free_Cash_Flow_Margin_pct
AMD,2024,1.641E9,3.041E9,0.118,0.824,1.853,6.36E8,2.405E9,0.093
AMD,2025,4.335E9,7.709E9,0.223,1.535,1.778,9.74E8,6.735E9,0.194
Adobe,2024,5.56E9,8.056E9,0.375,0.103,1.449,1.83E8,7.873E9,0.366
Adobe,2025,7.13E9,1.0031E10,0.422,0.245,1.407,1.79E8,9.852E9,0.414
Apple,2024,9.3736E10,1.18254E11,0.302,0.07,1.262,9.447E9,1.08807E11,0.278
Apple,2025,1.1201E11,1.11482E11,0.268,-0.057,0.995,1.2715E10,9.8767E10,0.237
CrowdStrike,2024,7.2181E7,1.166207E9,0.382,0.239,16.157,1.76529E8,9.89678E8,0.324
CrowdStrike,2025,-1.5241E7,1.381727E9,0.349,0.185,-90.659,2.54852E8,1.126875E9,0.285
CrowdStrike,2026,-1.62502E8,1.612349E9,0.335,0.167,-9.922,3.02108E8,1.310241E9,0.272
Microsoft,2024,8.8136E10,1.18548E11,0.484,0.354,1.345,4.4477E10,7.4071E10,0.302


## Financial Health & Risk

### Business Question

Does the company have the financial flexibility to support operations and growth if conditions tighten?

### Approach

View: `financial_health`

- Current_ratio
- debt_to_equity_ratio still treats a missing debt leg as 0. Short_Term_Debt_Reported and Long_Term_Debt_Reported record whether the source had a value.
- CrowdStrike short-term debt is unreported for FY2021 through FY2026 and is treated as 0.
- Liabilities_to_Assets. AMD null total liabilities are assets minus equity, because AMD companyfacts has no Total_Liabilities tag. Total_Liabilities_Imputed marks those rows. Other companies are not imputed.
- Primary ROA and ROE use average balances, (current + prior) / 2. Ending balances are used only when there is no prior year, and those rows are flagged.
- ROA_Ending and ROE_Ending keep the ending-balance ratios.
- Buybacks inflate Apple's ROE because they shrink stockholders' equity.


In [0]:
%sql
SELECT company,Fiscal_Year,Current_ratio,debt_to_equity_ratio,Liabilities_to_Assets,roa,roe
FROM financial_analytics.analytics.financial_health
where Fiscal_Year >=2024
ORDER BY company, Fiscal_Year;


company,Fiscal_Year,Current_ratio,debt_to_equity_ratio,Liabilities_to_Assets,roa,roe
AMD,2024,2.616,0.03,0.168,0.024,0.029
AMD,2025,2.85,0.051,0.181,0.056,0.069
Adobe,2024,1.068,0.399,0.533,0.184,0.394
Adobe,2025,0.996,0.534,0.606,0.242,0.613
Apple,2024,0.867,1.697,0.844,0.257,1.646
Apple,2025,0.893,1.23,0.795,0.312,1.519
CrowdStrike,2024,1.764,0.322,0.648,0.011,0.031
CrowdStrike,2025,1.766,0.227,0.619,-0.002,-0.005
CrowdStrike,2026,1.773,0.168,0.597,-0.015,-0.037
Microsoft,2024,1.275,0.167,0.476,0.172,0.328


## Peer Benchmarking

### Business question

How do the companies compare on the latest shared fiscal year?

### Approach

`peer_benchmarking` ranks nine FY2025 metrics with RANK(). FY2025 is the last year present for every company. The value is kept next to each rank.

Higher is better for revenue growth, operating margin, net income margin, OCF margin, current ratio, ROA, and ROE. Lower is better for operating expense / revenue and debt-to-equity. ROA and ROE are the primary average-balance returns.

Ties: RANK() shares the best rank and skips the next rank (pandas rank method='min').

Missing values: NULLS LAST, so a missing metric ranks after every reported value. Missing values tie with each other.

Average_Rank is the equal-weight average of those nine ranks. It is context for the section, not a single winner score. The six companies are in different technology subsectors and do not share one business model, so the average is a descriptive comparison, not a ranking of which company is best. Do not mix FY2025 and FY2026 in one ranking.

`executive_summary` attaches these FY2025 ranks on company alone, including when a company's latest year is FY2026. The columns are labeled FY2025_.


In [0]:
%sql
SELECT *
FROM financial_analytics.analytics.peer_benchmarking
ORDER BY Average_Rank


## Company analysis

Each read uses that company's latest fiscal year. Microsoft, NVIDIA, and CrowdStrike are FY2026. Apple, Adobe, and AMD are FY2025. The peer rank cited in each read is the FY2025 nine-metric average. A lower average rank is stronger. It is context, not a single score of business quality.


### Microsoft — FY2026

Revenue grew 17.8% in FY2026, up from 14.9% in FY2025. The FY2026 growth rate rose by 2.9 percentage points from the FY2025 rate. Revenue is still increasing.

Gross margin eased from 68.8% in FY2025 to 67.9% in FY2026. Operating profit moved the other way. The operating margin rose from 45.6% in FY2025 to 46.8% in FY2026, and the net income margin rose from 36.1% to 40.3%. Operating expenses were 21.2% of revenue in FY2026, down from 23.2% in FY2025 and 25.1% in FY2024. In FY2026, revenue growth exceeded operating-expense growth by 10.3 percentage points.

Operating cash flow grew 34.4% in FY2026. The operating-cash-flow margin rose from 48.3% in FY2025 to 55.1% in FY2026.

The balance sheet stayed conservative and leverage eased. The current ratio was 1.353 in FY2025 and 1.230 in FY2026. Debt-to-equity fell from 0.126 in FY2025 to 0.091 in FY2026, and liabilities-to-assets fell from 44.5% to 41.7%. Average-balance ROA rose from 18.0% in FY2025 to 19.4% in FY2026. Average-balance ROE rose from 33.3% to 34.0%.

Microsoft's FY2025 average rank was 3.111. That comparison uses FY2025 only.


### Apple — FY2025

Revenue grew 6.4% in FY2025, up from 2.0% in FY2024. The FY2025 growth rate improved by 4.4 percentage points from the FY2024 rate.

Gross margin rose from 46.2% in FY2024 to 46.9% in FY2025. The operating margin rose from 31.5% to 32.0%. Operating income grew 8.0% in FY2025. The net income margin rose from 24.0% in FY2024 to 26.9% in FY2025. Net income grew 19.5% in FY2025, from $93.7 billion in FY2024 to $112.0 billion in FY2025.

Operating expense growth exceeded revenue growth by 1.7 percentage points in FY2025. Operating expenses were 14.9% of revenue in FY2025, up from 14.7% in FY2024.

Operating cash flow fell 5.7% in FY2025, the same year net income grew. The operating-cash-flow margin fell from 30.2% in FY2024 to 26.8% in FY2025. Cash conversion was 0.995 in FY2025, down from 1.262 in FY2024. Both years are meaningful. Operating cash flow was about equal to net income in FY2025, after running above it in FY2024.

Debt-to-equity fell from 1.70 in FY2024 to 1.23 in FY2025, and liabilities-to-assets fell from 84.4% to 79.5%. The current ratio rose from 0.867 in FY2024 to 0.893 in FY2025 and remained below 1. Average-balance ROA rose from 26.1% in FY2024 to 30.9% in FY2025. Average-balance ROE rose from 157.4% to 171.4%. The high ROE is inflated by buybacks, which shrink equity.

Apple's FY2025 average rank was 4.000. In FY2025, Apple ranked 6th on revenue growth, 6th on the current ratio, and 6th on debt-to-equity, and 1st on ROE.


### NVIDIA — FY2026

Revenue grew 65.5% in FY2026, down from 114.2% in FY2025. The FY2026 growth rate fell by 48.7 percentage points from the FY2025 rate. Revenue is still growing. The slower rate is a cooldown from an unusually high FY2025, not a decline in revenue.

Gross margin was 71.1% in FY2026, after 75.0% in FY2025 and 72.7% in FY2024. The operating margin was 60.4% in FY2026, after 62.4% in FY2025 and 54.1% in FY2024. The net income margin was 55.6% in FY2026, compared with 55.8% in FY2025. Operating expenses fell from 12.6% of revenue in FY2025 to 10.7% in FY2026.

The operating-cash-flow margin was 47.6% in FY2026, down from 49.1% in FY2025, and operating cash flow grew 60.3% in FY2026. Cash generation is still strong. The margin is a little lower than the prior year.

The current ratio fell from 4.440 in FY2025 to 3.905 in FY2026 and remains high. Debt-to-equity fell from 0.107 to 0.054, and liabilities-to-assets fell from 28.9% to 23.9%. Average-balance ROA fell from 82.2% in FY2025 to 75.4% in FY2026. Ending-balance ROA fell from 65.3% to 58.1%. Those two ROA figures use different denominators. The primary figure is the average-balance return. Average-balance ROE fell from 119.2% in FY2025 to 101.5% in FY2026.

NVIDIA's FY2025 average rank was 1.222. That comparison uses FY2025 only.


### Adobe — FY2025

Revenue grew 10.5% in FY2025, compared with 10.8% in FY2024. The FY2025 growth rate eased by 0.3 percentage points from the FY2024 rate. Revenue is still growing.

Gross margin was 89.3% in FY2025, compared with 89.0% in FY2024. The operating margin rose from 31.3% in FY2024 to 36.6% in FY2025, and the net income margin rose from 25.9% to 30.0%. Operating-expense growth was 0.9% in FY2025, against FY2025 revenue growth of 10.5%. Operating expenses fell from 57.7% of revenue in FY2024 to 52.6% in FY2025.

The operating-cash-flow margin rose from 37.5% in FY2024 to 42.2% in FY2025, and operating cash flow grew 24.5% in FY2025.

The balance sheet moved the other way. The current ratio fell from 1.068 in FY2024 to 0.996 in FY2025. A current ratio slightly below 1.0 is a liquidity signal, not by itself a crisis. Debt-to-equity rose from 0.399 in FY2024 to 0.534 in FY2025, and liabilities-to-assets rose from 53.3% to 60.6%. Average-balance ROA rose from 18.5% in FY2024 to 23.9% in FY2025, and average-balance ROE rose from 36.3% to 55.4%.

Adobe's FY2025 average rank was 3.889, the same FY2025 average rank as AMD. That comparison uses FY2025 only.


### AMD — FY2025

Revenue growth accelerated from 13.7% in FY2024 to 34.3% in FY2025. The FY2025 growth rate rose by 20.6 percentage points from the FY2024 rate. Revenue is growing faster, not declining.

Gross margin was nearly unchanged, 49.4% in FY2024 and 49.5% in FY2025. The operating margin rose from 7.4% to 10.7%, and the net income margin rose from 6.4% to 12.5%. Operating income grew 94.4% in FY2025 after growing 373.8% in FY2024. Operating income is still rising. The lower FY2025 growth rate is not a decline in operating income. Operating expenses fell from 42.0% of revenue in FY2024 to 38.9% in FY2025.

The operating-cash-flow margin rose from 11.8% in FY2024 to 22.3% in FY2025, and operating cash flow grew 153.5% in FY2025.

The current ratio improved from 2.62 in FY2024 to 2.85 in FY2025. Debt-to-equity rose from 0.030 to 0.051. Liabilities-to-assets rose from 16.8% in FY2024 to 18.1% in FY2025. AMD does not report total liabilities, so that ratio uses assets minus equity. Average-balance ROA rose from 2.4% in FY2024 to 5.9% in FY2025, and average-balance ROE rose from 2.9% to 7.2%.

AMD's FY2025 average rank was 3.889, the same FY2025 average rank as Adobe. That comparison uses FY2025 only.


### CrowdStrike — FY2026

Revenue is still growing, and the rate has slowed. Growth was 36.3% in FY2024, 29.4% in FY2025, and 21.7% in FY2026. The slower rate is not a decline in revenue.

Gross margin was 75.2% in FY2024, 75.0% in FY2025, and 74.7% in FY2026. The operating margin went from −0.6% in FY2024 to −2.9% in FY2025 to −6.1% in FY2026. The net income margin went from 2.4% in FY2024 to −0.4% in FY2025 to −3.4% in FY2026.

Operating expenses were 75.8% of revenue in FY2024, 77.9% in FY2025, and 80.8% in FY2026. Operating-expense growth exceeded revenue growth by 4.5 percentage points in FY2026.

Operating cash flow stayed positive. The operating-cash-flow margin was 38.2% in FY2024, 34.9% in FY2025, and 33.5% in FY2026. Cash conversion is not meaningful because net income is negative. The FY2026 net-income growth rate is not a useful read of earnings, because the loss got larger.

The current ratio was 1.764 in FY2024, 1.766 in FY2025, and 1.773 in FY2026. Debt-to-equity fell from 0.322 in FY2024 to 0.227 in FY2025 to 0.168 in FY2026. Average-balance ROA went from 1.2% in FY2024 to −0.2% in FY2025 to −1.6% in FY2026.

CrowdStrike's FY2025 average rank was 4.889. That comparison uses FY2025 only.


## Management suggestions

These follow the company reads. Each item is one priority supported by that company's results. A possible explanation is not treated as an established cause.

**Microsoft.** Gross margin eased from 68.8% in FY2025 to 67.9% in FY2026, while the operating margin rose from 45.6% to 46.8% and operating expenses fell from 23.2% of revenue to 21.2%. Operating cash flow grew 34.4% in FY2026, and the operating-cash-flow margin rose from 48.3% to 55.1%. The operating-profit and operating-cash trends are improving. Gross margin is not. That matters because the higher operating margin depends on the lower expense ratio, not on a higher gross margin. Management should investigate what moved gross margin down in FY2026 and whether the lower expense ratio is what offset it. The results do not establish that cause.

**Apple.** Net income grew 19.5% in FY2025, from $93.7 billion in FY2024 to $112.0 billion in FY2025, while operating cash flow fell 5.7%. The operating-cash-flow margin fell from 30.2% in FY2024 to 26.8% in FY2025, and cash conversion fell from 1.262 to 0.995. The current ratio rose from 0.867 to 0.893 and stayed below 1. Debt-to-equity improved, from 1.70 in FY2024 to 1.23 in FY2025. Earnings and leverage moved in a favorable direction. Operating cash flow did not. That matters because reported earnings and cash generation moved in opposite directions in the same year. Management should investigate the gap between FY2025 net-income growth and the decline in operating cash flow, and what leaves the current ratio below 1. The results do not establish the cause.

**NVIDIA.** Revenue still grew 65.5% in FY2026. The growth rate was 48.7 percentage points lower than the 114.2% growth rate in FY2025. That is slower growth, not a decline in revenue. Gross margin fell from 75.0% in FY2025 to 71.1% in FY2026. The operating margin was 60.4% in FY2026, after 62.4% in FY2025 and 54.1% in FY2024. The operating-cash-flow margin eased from 49.1% in FY2025 to 47.6% in FY2026, and operating cash flow grew 60.3%. Average-balance ROA eased from 82.2% to 75.4%. The growth rate is decelerating, and gross margin is lower than in FY2025. Operating profit and operating cash flow are still strong. Management should investigate whether the gross margin, the operating margin, and the operating-cash-flow margin hold if revenue growth continues to cool. The results do not show that those margins will fall further.

**Adobe.** The operating margin rose from 31.3% in FY2024 to 36.6% in FY2025, and the operating-cash-flow margin rose from 37.5% to 42.2%. Operating expenses fell from 57.7% of revenue to 52.6%. Over the same year, the current ratio fell from 1.068 to 0.996, debt-to-equity rose from 0.399 to 0.534, and liabilities-to-assets rose from 53.3% to 60.6%. Profit and operating cash flow are improving. Liquidity and leverage are deteriorating. A current ratio slightly below 1.0 is a liquidity signal, not by itself a crisis. Management should investigate what changed in current assets, current liabilities, and debt. The results do not establish the cause.

**AMD.** Revenue growth accelerated from 13.7% in FY2024 to 34.3% in FY2025. The operating margin rose from 7.4% to 10.7%, the operating-cash-flow margin rose from 11.8% to 22.3%, and average-balance ROA rose from 2.4% to 5.9%. Liabilities-to-assets rose from 16.8% to 18.1%. That ratio uses assets minus equity, because AMD does not report total liabilities. Growth, margins, operating cash flow, and returns are improving. The liabilities-to-assets ratio moved the other way, from a low base. Management should investigate what accompanied the FY2025 acceleration, including the rise in liabilities-to-assets. The results do not establish the cause of that rise.

**CrowdStrike.** Operating expenses rose from 75.8% of revenue in FY2024 to 77.9% in FY2025 and 80.8% in FY2026. In FY2026, operating-expense growth exceeded revenue growth by 4.5 percentage points. The operating margin went from −0.6% in FY2024 to −6.1% in FY2026. Revenue was still growing, at 21.7% in FY2026. Operating cash flow stayed positive, and the operating-cash-flow margin eased from 34.9% in FY2025 to 33.5% in FY2026. The profitability trend is deteriorating. This is not a finding that operating cash flow turned negative. Management should investigate which operating costs grew faster than revenue. The results do not name those costs or establish why the gap opened.


## Conclusion

NVIDIA's FY2025 average rank was 1.222 and Microsoft's was 3.111. A lower average rank is stronger, and the rank is context rather than a single verdict. NVIDIA's FY2026 revenue growth of 65.5% is slower than 114.2% in FY2025, and revenue is still rising. Gross margin fell from 75.0% in FY2025 to 71.1% in FY2026, while the operating margin was still 60.4% and the operating-cash-flow margin was 47.6%. Microsoft's revenue growth was 17.8% in FY2026, up from 14.9% in FY2025. Gross margin eased from 68.8% to 67.9%, while the operating margin rose from 45.6% to 46.8% and the operating-cash-flow margin rose from 48.3% to 55.1%.

AMD and Adobe each have an FY2025 average rank of 3.889, with different latest-year patterns. AMD's revenue growth accelerated from 13.7% in FY2024 to 34.3% in FY2025, the operating margin rose from 7.4% to 10.7%, and operating cash flow grew 153.5%. Liabilities-to-assets rose from 16.8% to 18.1%, using imputed liabilities. Adobe's operating margin rose from 31.3% in FY2024 to 36.6% in FY2025 and operating expenses fell from 57.7% of revenue to 52.6%, while the current ratio fell from 1.068 to 0.996 and debt-to-equity rose from 0.399 to 0.534.

Apple's FY2025 revenue growth was 6.4%. Net income grew 19.5%, from $93.7 billion in FY2024 to $112.0 billion in FY2025, while operating cash flow fell 5.7% and cash conversion fell from 1.262 to 0.995. Debt-to-equity fell from 1.70 to 1.23, and the current ratio was 0.893. FY2025 average-balance ROE was 171.4%, inflated by buybacks that shrink equity. Apple's FY2025 average rank was 4.000. In that year it ranked 6th on revenue growth and 1st on ROE.

CrowdStrike's revenue is still growing, at 21.7% in FY2026 after 29.4% in FY2025 and 36.3% in FY2024. Gross margin was 74.7% in FY2026. The operating margin moved from −0.6% in FY2024 to −6.1% in FY2026, and operating expenses rose from 75.8% of revenue to 80.8%. Operating cash flow stayed positive, with a FY2026 margin of 33.5%. Its FY2025 average rank was 4.889.

Slower growth is not falling revenue. NVIDIA and CrowdStrike both grew more slowly in the latest year and still grew. The sharper differences are elsewhere. Microsoft's gross margin eased while its operating margin and operating cash flow improved. Apple's earnings rose while operating cash flow fell. Adobe's margins improved while its current ratio slipped below 1 and leverage rose. CrowdStrike's gross margin held near 75% while the operating margin became more negative and operating cash flow stayed positive.
